<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/OHDSI_Clean25_ReviewerResolution_COLAB_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OHDSI Clean 25-Patient Reviewer Resolution — Google Colab (Fixed)

This notebook is the **clean reproduction** that follows the historical-evidence audit.

The historical 25→27 discrepancy could not be causally reconstructed from preserved raw assets. Therefore this notebook does **not** guess a historical cause. Instead it creates a new, explicitly labeled, deterministic 25-patient synthetic reproduction in a **fresh SQLite database** and adds the checks the original demo should have had.

## Reviewer-resolution objectives

1. Generate a new deterministic 25-patient Synthea FHIR Bulk cohort.
2. Verify exactly 25 FHIR Patient resources before transformation.
3. Create a fresh, empty SQLite OMOP database.
4. Transform with `pyomop==6.4.0` and `fhiry==5.2.2`.
5. Assert whether FHIR Patient count equals OMOP PERSON count.
6. Recompute mapping completeness.
7. Diagnose drug and visit mapping behavior.
8. Search Google Drive for Athena vocabulary files.
9. Capture stage-level runtime and environment.
10. Produce GitHub-safe reviewer evidence plus private row-level diagnostics.

## Interpretation rule

This notebook is a **reproduction/correction run**, not a reconstruction of the original historical cohort.

If Athena files are unavailable, vocabulary-backed concept-mapping percentages are labeled **not fully reproducible in this run** rather than silently substituted.

### Fixed cohort-size control

This version explicitly runs Synthea with `-o false` (`overflowPopulation=false`).  
That prevents Synthea from exporting additional deceased overflow patients while generating replacements to reach the requested living population. The reviewer-resolution source gate therefore targets an **exact 25 exported Patient resources / 25 unique Patient IDs**.


## 0. Install pinned project dependencies and clone the repository

In [1]:
import sys, subprocess, time
from pathlib import Path

REPO_URL = "https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git"
REPO_DIR = Path("/content/ohdsi-fhir-omop-showcase-demo")

install_started = time.perf_counter()
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "pyomop==6.4.0",
        "fhiry==5.2.2",
        "nest-asyncio",
        "psutil",
        "pyarrow",
        "tabulate",
    ],
    check=True,
)
INSTALL_SECONDS = time.perf_counter() - install_started

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

print("Repository:", REPO_DIR)
print("Dependency install seconds:", round(INSTALL_SECONDS, 3))

Repository: /content/ohdsi-fhir-omop-showcase-demo
Dependency install seconds: 4.818


## 1. Mount Google Drive and create isolated run directories

In [2]:
import os, json, shutil, sqlite3, platform, hashlib, tempfile, asyncio, re, gzip
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")
PRIVATE_ROOT = MYDRIVE / "fhir_omop_colab" / "reviewer_clean25"
FHIR_OUTPUT_ROOT = PRIVATE_ROOT / "synthea_output"
OMOP_PRIVATE_DIR = PRIVATE_ROOT / "omop"
PRIVATE_DIAG_DIR = PRIVATE_ROOT / "private_diagnostics"
PUBLIC_OUT = REPO_DIR / "results" / "reviewer_clean25_resolution"

for p in [PRIVATE_ROOT, FHIR_OUTPUT_ROOT, OMOP_PRIVATE_DIR, PRIVATE_DIAG_DIR, PUBLIC_OUT]:
    p.mkdir(parents=True, exist_ok=True)

DB_PATH = OMOP_PRIVATE_DIR / "reviewer_clean25.sqlite"

SYNTHEA_SEED = 20260815
SYNTHEA_REFERENCE_DATE = "20260815"
SYNTHEA_POPULATION = 25
SYNTHEA_STATE = "Massachusetts"

print("PRIVATE_ROOT:", PRIVATE_ROOT)
print("PUBLIC_OUT:", PUBLIC_OUT)
print("DB_PATH:", DB_PATH)

Mounted at /content/drive
PRIVATE_ROOT: /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25
PUBLIC_OUT: /content/ohdsi-fhir-omop-showcase-demo/results/reviewer_clean25_resolution
DB_PATH: /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/omop/reviewer_clean25.sqlite


## 2. Environment manifest

In [3]:
import pyomop
import fhiry

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

PYOMOP_DIR = Path(pyomop.__file__).resolve().parent
MAPPING_PATH = PYOMOP_DIR / "mapping.default.json"

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "logical_cpu_count": psutil.cpu_count(logical=True),
    "physical_cpu_count": psutil.cpu_count(logical=False),
    "ram_gb": round(psutil.virtual_memory().total / (1024**3), 2),
    "pyomop": getattr(pyomop, "__version__", "unknown"),
    "fhiry": getattr(fhiry, "__version__", "unknown"),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "mapping_path": str(MAPPING_PATH),
    "mapping_sha256": sha256_file(MAPPING_PATH) if MAPPING_PATH.exists() else None,
    "dependency_install_seconds": round(INSTALL_SECONDS, 4),
}
display(pd.DataFrame([environment]).T)

,0
python,"3.12.13 (main, Mar 4 2026, 09:23:07) [GCC 11...."
platform,Linux-6.6.122+-x86_64-with-glibc2.35
logical_cpu_count,8
physical_cpu_count,4
ram_gb,50.99
pyomop,6.4.0
fhiry,5.2.2
pandas,2.2.2
numpy,2.0.2
mapping_path,/usr/local/lib/python3.12/dist-packages/pyomop...


# Part A — Generate the new deterministic 25-patient Synthea cohort

## 3. Clone Synthea and record the exact commit

In [4]:
SYNTHEA_DIR = Path("/content/synthea")

if SYNTHEA_DIR.exists():
    shutil.rmtree(SYNTHEA_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/synthetichealth/synthea.git", str(SYNTHEA_DIR)],
    check=True,
)

synthea_sha = subprocess.run(
    ["git", "-C", str(SYNTHEA_DIR), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True
).stdout.strip()

print("Synthea commit:", synthea_sha)

Synthea commit: 7e08387c68a7f0e21d13076609a159fd473fc902


## 4. Generate exactly 25 synthetic patients

Before generation, the run-specific Synthea output directory is deleted to prevent contamination from any earlier execution.

This fixed version also uses:

```text
-o false
```

so the exported cohort is constrained to the requested population size rather than including deceased overflow patients plus replacement patients.


In [5]:
if FHIR_OUTPUT_ROOT.exists():
    shutil.rmtree(FHIR_OUTPUT_ROOT)
FHIR_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

synthea_cmd = [
    "./run_synthea",
    "-s", str(SYNTHEA_SEED),
    "-p", str(SYNTHEA_POPULATION),
    "-o", "false",
    "-r", SYNTHEA_REFERENCE_DATE,
    "--exporter.fhir.export=true",
    "--exporter.fhir.bulk_data=true",
    f"--exporter.baseDirectory={FHIR_OUTPUT_ROOT}",
    SYNTHEA_STATE,
]

print("Running:")
print(" ".join(synthea_cmd))

generation_started = time.perf_counter()
proc = subprocess.run(
    synthea_cmd,
    cwd=str(SYNTHEA_DIR),
    capture_output=True,
    text=True,
)
SYNTHEA_GENERATION_SECONDS = time.perf_counter() - generation_started

print("Return code:", proc.returncode)
print("Generation seconds:", round(SYNTHEA_GENERATION_SECONDS, 3))
print("\nstdout tail:")
print(proc.stdout[-5000:])
if proc.stderr.strip():
    print("\nstderr tail:")
    print(proc.stderr[-5000:])

(PRIVATE_DIAG_DIR / "synthea_generation_stdout_stderr.txt").write_text(
    "COMMAND\n" + " ".join(synthea_cmd)
    + "\n\nSTDOUT\n" + proc.stdout
    + "\n\nSTDERR\n" + proc.stderr,
    encoding="utf-8"
)

if proc.returncode != 0:
    raise RuntimeError("Synthea generation failed. Inspect the saved private log.")

Running:
./run_synthea -s 20260815 -p 25 -o false -r 20260815 --exporter.fhir.export=true --exporter.fhir.bulk_data=true --exporter.baseDirectory=/content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output Massachusetts
Return code: 0
Generation seconds: 52.413

stdout tail:
.json
Loading module modules/hypothyroidism.json
Loading module modules/injuries.json
Loading module modules/kidney_transplant.json
Loading module modules/lung_cancer.json
Loading module modules/lupus.json
Loading module modules/mTBI.json
Loading module modules/med_rec.json
Loading module modules/mend_program.json
Loading module modules/metabolic_syndrome_care.json
Loading module modules/metabolic_syndrome_disease.json
Loading module modules/myocardial_infarction.json
Loading module modules/opioid_addiction.json
Loading module modules/osteoarthritis.json
Loading module modules/osteoporosis.json
Loading module modules/pregnancy.json
Loading module modules/prescribing_opioids_for_chronic_pain_and_treatment

## 5. Locate Bulk FHIR NDJSON files and assert exactly 25 Patient resources

The run must pass all of the following before any OMOP transformation occurs:

- 25 exported FHIR `Patient` resources
- 25 unique non-null Patient IDs
- 0 duplicate Patient IDs

If the gate fails, execution stops rather than silently changing the expected cohort size.


In [6]:
def iter_ndjson_resources(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue
            if isinstance(obj, dict) and obj.get("resourceType"):
                yield obj

ndjson_files = sorted(
    list(FHIR_OUTPUT_ROOT.rglob("*.ndjson"))
    + list(FHIR_OUTPUT_ROOT.rglob("*.ndjson.gz"))
)

print("NDJSON files found:", len(ndjson_files))
for p in ndjson_files:
    print(" -", p)

resource_counts = Counter()
patient_ids = []

for p in ndjson_files:
    for res in iter_ndjson_resources(p):
        rt = res.get("resourceType", "UNKNOWN")
        resource_counts[rt] += 1
        if rt == "Patient":
            patient_ids.append(str(res.get("id")) if res.get("id") is not None else None)

patient_ids_clean = [x for x in patient_ids if x]
patient_unique = sorted(set(patient_ids_clean))

resource_inventory_df = pd.DataFrame(
    sorted(resource_counts.items()),
    columns=["resource_type", "resource_count"]
)
display(resource_inventory_df)

source_gate = pd.DataFrame([{
    "expected_patient_resources": SYNTHEA_POPULATION,
    "observed_patient_resources": len(patient_ids),
    "unique_nonnull_patient_ids": len(patient_unique),
    "duplicate_patient_id_rows": len(patient_ids_clean) - len(patient_unique),
    "passed_exact_25_gate": (
        len(patient_ids) == SYNTHEA_POPULATION
        and len(patient_unique) == SYNTHEA_POPULATION
    )
}])
display(source_gate)

resource_inventory_df.to_csv(PUBLIC_OUT / "clean25_fhir_resource_inventory.csv", index=False)
source_gate.to_csv(PUBLIC_OUT / "clean25_source_patient_gate.csv", index=False)

if not bool(source_gate.iloc[0]["passed_exact_25_gate"]):
    raise RuntimeError(
        f"Source cohort gate failed: expected 25 unique Patient resources, "
        f"observed {len(patient_ids)} resources / {len(patient_unique)} unique IDs."
    )

pd.DataFrame({"fhir_patient_id": patient_unique}).to_csv(
    PRIVATE_DIAG_DIR / "clean25_fhir_patient_ids_PRIVATE.csv",
    index=False
)

print("PASS: clean source contains exactly 25 unique Patient resources.")

NDJSON files found: 24
 - /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir/AllergyIntolerance.ndjson
 - /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir/CarePlan.ndjson
 - /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir/CareTeam.ndjson
 - /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir/Claim.ndjson
 - /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir/Condition.ndjson
 - /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir/Device.ndjson
 - /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir/DiagnosticReport.ndjson
 - /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir/DocumentReference.ndjson
 - /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir/Encounter.ndjson
 - /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir/ExplanationOfBenefi

,resource_type,resource_count
0,AllergyIntolerance,31
1,CarePlan,76
2,CareTeam,76
3,Claim,3134
4,Condition,791
5,Device,120
6,DiagnosticReport,2580
7,DocumentReference,1324
8,Encounter,1324
9,ExplanationOfBenefit,3134


,expected_patient_resources,observed_patient_resources,unique_nonnull_patient_ids,duplicate_patient_id_rows,passed_exact_25_gate
0,25,25,25,0,True


PASS: clean source contains exactly 25 unique Patient resources.


# Part B — FHIRy normalization and fresh pyOMOP transformation

## 6. Normalize Bulk NDJSON with the same FHIRy path used by the project

In [7]:
import fhiry.parallel as fp

FHIRY_CONFIG = {"REMOVE": ["text.div", "meta"], "RENAME": {}}

ndjson_parent_counts = Counter(str(p.parent) for p in ndjson_files)
FHIR_BULK_DIR = Path(ndjson_parent_counts.most_common(1)[0][0])

print("FHIR_BULK_DIR:", FHIR_BULK_DIR)

normalize_started = time.perf_counter()
source_df = fp.ndjson(
    str(FHIR_BULK_DIR),
    config_json=json.dumps(FHIRY_CONFIG)
)
FHIRY_NORMALIZATION_SECONDS = time.perf_counter() - normalize_started

print("Normalized rows:", len(source_df))
print("Columns:", len(source_df.columns))
print("Normalization seconds:", round(FHIRY_NORMALIZATION_SECONDS, 3))

RESOURCE_TYPE_CANDIDATES = ["resourceType", "resource.resourceType"]
rt_col = next((c for c in RESOURCE_TYPE_CANDIDATES if c in source_df.columns), None)
if rt_col is None:
    raise KeyError("FHIR resource type column not found after FHIRy normalization.")

normalized_inventory_df = (
    source_df.groupby(rt_col, dropna=False)
    .size()
    .rename("normalized_rows")
    .reset_index()
    .rename(columns={rt_col: "resource_type"})
    .sort_values("normalized_rows", ascending=False)
)
display(normalized_inventory_df)
normalized_inventory_df.to_csv(
    PUBLIC_OUT / "clean25_fhiry_normalized_inventory.csv",
    index=False
)

FHIR_BULK_DIR: /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir


Processing NDJSON files: 100%|██████████| 24/24 [01:57<00:00,  4.89s/it]


Normalized rows: 34337
Columns: 155
Normalization seconds: 117.581


,resource_type,normalized_rows
16,Observation,11650
21,Procedure,4370
3,Claim,3134
9,ExplanationOfBenefit,3134
6,DiagnosticReport,2580
15,MedicationRequest,1810
7,DocumentReference,1324
8,Encounter,1324
14,MedicationAdministration,1228
13,Medication,1228


## 7. Search Google Drive for Athena vocabulary files

In [8]:
ATHENA_REQUIRED = {"concept", "concept_relationship", "vocabulary"}

def normalized_vocab_filename(name):
    low = name.lower()
    for suffix in [".csv", ".csv.gz", ".txt", ".txt.gz"]:
        if low.endswith(suffix):
            return low[:-len(suffix)]
    return low

def find_athena_dirs(root, max_hits=30):
    hits = []
    if not Path(root).exists():
        return hits

    prune = {
        ".git", "node_modules", ".venv", "venv", "__pycache__",
        "site-packages", ".Trash"
    }

    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in prune]
        names = {normalized_vocab_filename(f) for f in files}
        if ATHENA_REQUIRED.issubset(names):
            hits.append(Path(current))
            if len(hits) >= max_hits:
                break
    return hits

athena_candidates = find_athena_dirs(MYDRIVE)
ATHENA_VOCAB_DIR = athena_candidates[0] if len(athena_candidates) == 1 else None

athena_df = pd.DataFrame([{
    "candidate_path": str(p),
    "auto_selected": ATHENA_VOCAB_DIR is not None and p == ATHENA_VOCAB_DIR
} for p in athena_candidates])

print("Athena candidates:", len(athena_candidates))
if not athena_df.empty:
    display(athena_df)

athena_df.to_csv(PUBLIC_OUT / "clean25_athena_candidates.csv", index=False)

if ATHENA_VOCAB_DIR:
    print("Athena auto-selected:", ATHENA_VOCAB_DIR)
elif len(athena_candidates) > 1:
    print("Multiple Athena candidates found; no directory auto-selected.")
else:
    print("No Athena vocabulary directory found. Vocabulary-backed mapping reproduction is gated.")

Athena candidates: 0
No Athena vocabulary directory found. Vocabulary-backed mapping reproduction is gated.


## 8. Create a guaranteed fresh SQLite database and run pyOMOP

In [10]:
from pyomop import CdmEngineFactory
from pyomop.cdm54 import Base
from pyomop.loader import CdmCsvLoader
from pyomop.vocabulary import CdmVocabulary

import nest_asyncio
import asyncio
import tempfile
import time
from pathlib import Path

nest_asyncio.apply()

if not MAPPING_PATH.exists():
    raise FileNotFoundError(f"pyOMOP mapping file missing: {MAPPING_PATH}")

# ------------------------------------------------------------
# 1. GUARANTEE A FRESH DATABASE
# ------------------------------------------------------------
db_existed_before = DB_PATH.exists()

if DB_PATH.exists():
    DB_PATH.unlink()

DB_PATH.parent.mkdir(parents=True, exist_ok=True)

assert not DB_PATH.exists(), (
    f"Fresh-database gate failed: {DB_PATH} still exists."
)

print("Fresh SQLite target:", DB_PATH)

# ------------------------------------------------------------
# 2. PYOMOP LOAD FUNCTION
# ------------------------------------------------------------
async def clean25_pyomop_load(df, db_path, vocab_dir=None):

    # Configure pyOMOP
    cdm = CdmEngineFactory(
        db="sqlite",
        name=str(db_path),
    )

    # IMPORTANT:
    # CdmEngineFactory initializes the SQLAlchemy engine lazily.
    # Accessing .engine creates it before init_models().
    engine = cdm.engine

    if engine is None:
        raise RuntimeError(
            "pyOMOP failed to initialize the SQLite engine."
        )

    print("pyOMOP engine initialized:", engine)

    # Create fresh OMOP CDM 5.4 tables
    schema_started = time.perf_counter()
    await cdm.init_models(Base.metadata)
    schema_seconds = time.perf_counter() - schema_started

    print(
        "OMOP CDM 5.4 schema initialized in",
        round(schema_seconds, 3),
        "seconds"
    )

    # --------------------------------------------------------
    # Optional Athena vocabulary load
    # --------------------------------------------------------
    vocab_seconds = None

    if vocab_dir is not None:

        vocab_dir = Path(vocab_dir)

        if not vocab_dir.exists():
            raise FileNotFoundError(
                f"Athena vocabulary directory does not exist: {vocab_dir}"
            )

        print("Loading Athena vocabulary from:", vocab_dir)

        vocab_started = time.perf_counter()

        vocab = CdmVocabulary(
            cdm,
            version="cdm54"
        )

        await vocab.create_vocab(str(vocab_dir))

        vocab_seconds = time.perf_counter() - vocab_started

        print(
            "Athena vocabulary loaded in",
            round(vocab_seconds, 3),
            "seconds"
        )

    else:
        print(
            "Athena vocabulary not available — "
            "continuing with structural FHIR→OMOP transformation only."
        )

    # --------------------------------------------------------
    # 3. WRITE NORMALIZED FHIR DATA TO TEMPORARY CSV
    # --------------------------------------------------------
    with tempfile.NamedTemporaryFile(
        suffix=".csv",
        delete=False
    ) as tmp:
        temp_csv = Path(tmp.name)

    try:

        csv_started = time.perf_counter()

        df.to_csv(
            temp_csv,
            index=False
        )

        csv_seconds = time.perf_counter() - csv_started

        print(
            "Temporary normalized CSV created:",
            temp_csv
        )
        print(
            "CSV write seconds:",
            round(csv_seconds, 3)
        )

        # ----------------------------------------------------
        # 4. RUN PYOMOP TRANSFORMATION
        # ----------------------------------------------------
        loader = CdmCsvLoader(
            cdm,
            version="cdm54"
        )

        transform_started = time.perf_counter()

        await loader.load(
            csv_path=str(temp_csv),
            mapping_path=str(MAPPING_PATH),
            chunk_size=500,
        )

        transform_seconds = (
            time.perf_counter() - transform_started
        )

        print(
            "pyOMOP transformation completed in",
            round(transform_seconds, 3),
            "seconds"
        )

    finally:

        if temp_csv.exists():
            temp_csv.unlink()

        await cdm.dispose()

    return (
        schema_seconds,
        vocab_seconds,
        transform_seconds,
    )


# ------------------------------------------------------------
# 5. EXECUTE
# ------------------------------------------------------------
load_started = time.perf_counter()

(
    OMOP_SCHEMA_SECONDS,
    VOCAB_LOAD_SECONDS,
    PYOMOP_TRANSFORM_SECONDS,
) = asyncio.get_event_loop().run_until_complete(
    clean25_pyomop_load(
        source_df,
        DB_PATH,
        ATHENA_VOCAB_DIR
    )
)

TOTAL_OMOP_LOAD_SECONDS = (
    time.perf_counter() - load_started
)

# ------------------------------------------------------------
# 6. VERIFY DATABASE EXISTS
# ------------------------------------------------------------
if not DB_PATH.exists():
    raise RuntimeError(
        f"pyOMOP completed but SQLite database was not created: {DB_PATH}"
    )

if DB_PATH.stat().st_size == 0:
    raise RuntimeError(
        f"SQLite database exists but is empty: {DB_PATH}"
    )

print("\n" + "=" * 70)
print("CLEAN OMOP LOAD COMPLETE")
print("=" * 70)

print("DB existed before cell:", db_existed_before)
print(
    "Fresh DB size MB:",
    round(DB_PATH.stat().st_size / (1024 ** 2), 3)
)
print(
    "OMOP schema seconds:",
    round(OMOP_SCHEMA_SECONDS, 3)
)
print(
    "Vocabulary loaded:",
    ATHENA_VOCAB_DIR is not None
)
print(
    "Vocabulary seconds:",
    None
    if VOCAB_LOAD_SECONDS is None
    else round(VOCAB_LOAD_SECONDS, 3)
)
print(
    "pyOMOP transformation seconds:",
    round(PYOMOP_TRANSFORM_SECONDS, 3)
)
print(
    "Total OMOP load seconds:",
    round(TOTAL_OMOP_LOAD_SECONDS, 3)
)
print("=" * 70)

Fresh SQLite target: /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/omop/reviewer_clean25.sqlite
pyOMOP engine initialized: <sqlalchemy.ext.asyncio.engine.AsyncEngine object at 0x7c162bfa1f90>
OMOP CDM 5.4 schema initialized in 1.685 seconds
Athena vocabulary not available — continuing with structural FHIR→OMOP transformation only.
Temporary normalized CSV created: /tmp/tmposnt1t5b.csv
CSV write seconds: 1.766
pyOMOP transformation completed in 47.473 seconds

CLEAN OMOP LOAD COMPLETE
DB existed before cell: False
Fresh DB size MB: 6.039
OMOP schema seconds: 1.685
Vocabulary loaded: False
Vocabulary seconds: None
pyOMOP transformation seconds: 47.473
Total OMOP load seconds: 50.971


# Part C — Direct 25→PERSON reconciliation

## 9. Count OMOP PERSON rows and inspect source provenance

In [11]:
def table_exists(conn, table):
    return conn.execute(
        "SELECT 1 FROM sqlite_master WHERE type='table' AND name=? LIMIT 1",
        (table,)
    ).fetchone() is not None

def columns_for(conn, table):
    return [r[1] for r in conn.execute(f'PRAGMA table_info("{table}")').fetchall()]

conn = sqlite3.connect(str(DB_PATH))

if not table_exists(conn, "person"):
    raise RuntimeError("OMOP PERSON table not found.")

person_cols = columns_for(conn, "person")
person_count = int(conn.execute("SELECT COUNT(*) FROM person").fetchone()[0])

select_cols = [
    c for c in [
        "person_id", "person_source_value", "gender_concept_id",
        "year_of_birth", "gender_source_value"
    ] if c in person_cols
]
person_df = pd.read_sql_query(
    "SELECT " + ", ".join([f'"{c}"' for c in select_cols]) + " FROM person",
    conn
)

direct_count_gate = person_count == len(patient_unique)

reconciliation = {
    "fhir_patient_resources": len(patient_ids),
    "fhir_unique_patient_ids": len(patient_unique),
    "omop_person_rows": person_count,
    "count_reconciliation_pass": direct_count_gate,
    "person_source_value_available": "person_source_value" in person_df.columns,
}

if "person_source_value" in person_df.columns:
    omop_source_values = set(
        x for x in person_df["person_source_value"].dropna().astype(str).str.strip()
        if x
    )
    fhir_ids = set(patient_unique)
    reconciliation.update({
        "unique_nonblank_person_source_values": len(omop_source_values),
        "direct_id_matches": len(fhir_ids & omop_source_values),
        "fhir_ids_not_in_person_source_value": len(fhir_ids - omop_source_values),
        "person_source_values_not_in_fhir_ids": len(omop_source_values - fhir_ids),
    })

    pd.DataFrame({
        "omop_person_source_value": sorted(omop_source_values)
    }).to_csv(
        PRIVATE_DIAG_DIR / "clean25_person_source_values_PRIVATE.csv",
        index=False
    )

reconciliation_df = pd.DataFrame([reconciliation])
display(reconciliation_df)
reconciliation_df.to_csv(
    PUBLIC_OUT / "clean25_person_reconciliation.csv",
    index=False
)

print("RESULT:", f"{len(patient_unique)} unique FHIR Patients -> {person_count} OMOP PERSON rows")

if direct_count_gate:
    print("PASS: fresh current workflow has exact source-to-PERSON row-count reconciliation.")
else:
    print("REVIEW: fresh workflow still produces a PERSON discrepancy; this run is now directly diagnosable.")

,fhir_patient_resources,fhir_unique_patient_ids,omop_person_rows,count_reconciliation_pass,person_source_value_available,unique_nonblank_person_source_values,direct_id_matches,fhir_ids_not_in_person_source_value,person_source_values_not_in_fhir_ids
0,25,25,25,True,True,25,25,0,0


RESULT: 25 unique FHIR Patients -> 25 OMOP PERSON rows
PASS: fresh current workflow has exact source-to-PERSON row-count reconciliation.


## 10. Clinical OMOP row counts

In [12]:
CLINICAL_TABLES = [
    "person",
    "visit_occurrence",
    "condition_occurrence",
    "drug_exposure",
    "measurement",
    "observation",
    "procedure_occurrence",
]

count_rows = []
for table in CLINICAL_TABLES:
    if table_exists(conn, table):
        count_rows.append({
            "omop_table": table,
            "row_count": int(conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0])
        })

omop_counts_df = pd.DataFrame(count_rows)
display(omop_counts_df)
omop_counts_df.to_csv(PUBLIC_OUT / "clean25_omop_table_counts.csv", index=False)

,omop_table,row_count
0,person,25
1,visit_occurrence,1324
2,condition_occurrence,791
3,drug_exposure,2213
4,measurement,11650
5,observation,11681
6,procedure_occurrence,4370


# Part D — Mapping completeness and root-cause diagnostics

## 11. Recompute mapping completeness

In [13]:
MAPPING_CONFIG = {
    "condition_occurrence": ("condition_concept_id", "condition_source_value", "condition_source_concept_id"),
    "drug_exposure": ("drug_concept_id", "drug_source_value", "drug_source_concept_id"),
    "measurement": ("measurement_concept_id", "measurement_source_value", "measurement_source_concept_id"),
    "observation": ("observation_concept_id", "observation_source_value", "observation_source_concept_id"),
    "visit_occurrence": ("visit_concept_id", "visit_source_value", "visit_source_concept_id"),
}

mapping_rows = []

for table, (concept_col, source_value_col, source_concept_col) in MAPPING_CONFIG.items():
    if not table_exists(conn, table):
        continue

    cols = columns_for(conn, table)
    if concept_col not in cols:
        continue

    total, mapped = conn.execute(
        f'''SELECT
               COUNT(*),
               SUM(CASE WHEN "{concept_col}" IS NOT NULL AND "{concept_col}" <> 0 THEN 1 ELSE 0 END)
            FROM "{table}"'''
    ).fetchone()

    total = int(total or 0)
    mapped = int(mapped or 0)

    mapping_rows.append({
        "omop_table": table,
        "concept_column": concept_col,
        "total_records": total,
        "mapped_records": mapped,
        "unmapped_records": total - mapped,
        "mapped_percent": round((100.0 * mapped / total), 4) if total else np.nan,
        "athena_loaded_in_this_run": ATHENA_VOCAB_DIR is not None,
        "comparison_to_submitted_mapping_valid": ATHENA_VOCAB_DIR is not None,
    })

mapping_df = pd.DataFrame(mapping_rows)
display(mapping_df)
mapping_df.to_csv(
    PUBLIC_OUT / "clean25_mapping_completeness.csv",
    index=False
)

,omop_table,concept_column,total_records,mapped_records,unmapped_records,mapped_percent,athena_loaded_in_this_run,comparison_to_submitted_mapping_valid
0,condition_occurrence,condition_concept_id,791,0,791,0.0,False,False
1,drug_exposure,drug_concept_id,2213,0,2213,0.0,False,False
2,measurement,measurement_concept_id,11650,0,11650,0.0,False,False
3,observation,observation_concept_id,11681,0,11681,0.0,False,False
4,visit_occurrence,visit_concept_id,1324,0,1324,0.0,False,False


## 12. Inspect pyOMOP's active Encounter and medication mapping rules

In [14]:
with open(MAPPING_PATH, "r", encoding="utf-8") as f:
    pyomop_mapping = json.load(f)

rule_rows = []
for idx, table_map in enumerate(pyomop_mapping.get("tables", []), start=1):
    filters = table_map.get("filters", []) or []
    resource_type = None
    for flt in filters:
        if flt.get("column") in {"resourceType", "resource.resourceType"} and "equals" in flt:
            resource_type = str(flt["equals"])
            break

    if resource_type in {
        "Encounter", "MedicationRequest", "MedicationStatement",
        "Medication", "Immunization"
    }:
        rule_rows.append({
            "rule_order": idx,
            "source_resource_type": resource_type,
            "target_omop_table": table_map.get("name"),
            "source_paths_used": "|".join(
                sorted({
                    v for v in (table_map.get("columns", {}) or {}).values()
                    if isinstance(v, str) and v
                })
            ),
            "filters_json": json.dumps(filters, sort_keys=True),
        })

mapping_rules_df = pd.DataFrame(rule_rows)
display(mapping_rules_df)
mapping_rules_df.to_csv(
    PUBLIC_OUT / "clean25_relevant_pyomop_mapping_rules.csv",
    index=False
)

,rule_order,source_resource_type,target_omop_table,source_paths_used,filters_json
0,2,Encounter,visit_occurrence,class.code|hospitalization.dischargeDispositio...,"[{""column"": ""resourceType"", ""equals"": ""Encount..."
1,9,Immunization,drug_exposure,occurrenceDateTime|patientId|vaccineCode.codin...,"[{""column"": ""resourceType"", ""equals"": ""Immuniz..."
2,10,MedicationRequest,drug_exposure,authoredOn|dosageInstruction|medicationCodeabl...,"[{""column"": ""resourceType"", ""equals"": ""Medicat..."


## 13. Drug and visit unmapped-row diagnostics

In [15]:
gap_rows = []

for table in ["drug_exposure", "visit_occurrence"]:
    if table not in MAPPING_CONFIG or not table_exists(conn, table):
        continue

    concept_col, source_value_col, source_concept_col = MAPPING_CONFIG[table]
    cols = columns_for(conn, table)

    if concept_col not in cols:
        continue

    total_unmapped = int(conn.execute(
        f'SELECT COUNT(*) FROM "{table}" WHERE "{concept_col}" IS NULL OR "{concept_col}" = 0'
    ).fetchone()[0])

    row = {
        "omop_table": table,
        "unmapped_records": total_unmapped,
        "source_value_column_available": source_value_col in cols,
        "source_concept_column_available": source_concept_col in cols,
    }

    if source_value_col in cols:
        blank, nonblank = conn.execute(
            f'''SELECT
                   SUM(CASE WHEN "{source_value_col}" IS NULL OR TRIM(CAST("{source_value_col}" AS TEXT)) = '' THEN 1 ELSE 0 END),
                   SUM(CASE WHEN "{source_value_col}" IS NOT NULL AND TRIM(CAST("{source_value_col}" AS TEXT)) <> '' THEN 1 ELSE 0 END)
                FROM "{table}"
                WHERE "{concept_col}" IS NULL OR "{concept_col}" = 0'''
        ).fetchone()
        row["unmapped_blank_source_value"] = int(blank or 0)
        row["unmapped_nonblank_source_value"] = int(nonblank or 0)

        top = pd.read_sql_query(
            f'''SELECT CAST("{source_value_col}" AS TEXT) AS source_value, COUNT(*) AS records
                FROM "{table}"
                WHERE "{concept_col}" IS NULL OR "{concept_col}" = 0
                GROUP BY CAST("{source_value_col}" AS TEXT)
                ORDER BY records DESC
                LIMIT 100''',
            conn
        )
        top.to_csv(
            PRIVATE_DIAG_DIR / f"{table}_unmapped_source_values_PRIVATE.csv",
            index=False
        )

    if source_concept_col in cols:
        zero_sc, nonzero_sc = conn.execute(
            f'''SELECT
                   SUM(CASE WHEN "{source_concept_col}" IS NULL OR "{source_concept_col}" = 0 THEN 1 ELSE 0 END),
                   SUM(CASE WHEN "{source_concept_col}" IS NOT NULL AND "{source_concept_col}" <> 0 THEN 1 ELSE 0 END)
                FROM "{table}"
                WHERE "{concept_col}" IS NULL OR "{concept_col}" = 0'''
        ).fetchone()
        row["unmapped_zero_or_null_source_concept_id"] = int(zero_sc or 0)
        row["unmapped_nonzero_source_concept_id"] = int(nonzero_sc or 0)

    gap_rows.append(row)

gap_df = pd.DataFrame(gap_rows)
display(gap_df)
gap_df.to_csv(
    PUBLIC_OUT / "clean25_drug_visit_gap_diagnostics.csv",
    index=False
)

,omop_table,unmapped_records,source_value_column_available,source_concept_column_available,unmapped_blank_source_value,unmapped_nonblank_source_value,unmapped_zero_or_null_source_concept_id,unmapped_nonzero_source_concept_id
0,drug_exposure,2213,True,True,1228,985,2213,0
1,visit_occurrence,1324,True,True,0,1324,1324,0


## 14. Raw FHIR coding diagnostics for Encounter, MedicationRequest, Medication, and Immunization

In [16]:
coding_rows = []

def iter_codings(codeable):
    if isinstance(codeable, dict):
        for coding in codeable.get("coding", []) or []:
            if isinstance(coding, dict):
                yield coding

def add_coding(rt, field, coding):
    coding_rows.append({
        "resource_type": rt,
        "field": field,
        "system": coding.get("system"),
        "code": coding.get("code"),
        "display": coding.get("display"),
    })

for p in ndjson_files:
    for res in iter_ndjson_resources(p):
        rt = res.get("resourceType")

        if rt == "Encounter":
            enc_class = res.get("class")
            if isinstance(enc_class, dict):
                if "coding" in enc_class:
                    for c in iter_codings(enc_class):
                        add_coding(rt, "class", c)
                elif "code" in enc_class:
                    add_coding(rt, "class", enc_class)
            elif isinstance(enc_class, list):
                for item in enc_class:
                    if isinstance(item, dict) and "coding" in item:
                        for c in iter_codings(item):
                            add_coding(rt, "class", c)
                    elif isinstance(item, dict) and "code" in item:
                        add_coding(rt, "class", item)

            for i, typ in enumerate(res.get("type", []) or []):
                for c in iter_codings(typ):
                    add_coding(rt, f"type[{i}]", c)

        elif rt in {"MedicationRequest", "MedicationStatement"}:
            for c in iter_codings(res.get("medicationCodeableConcept")):
                add_coding(rt, "medicationCodeableConcept", c)

        elif rt == "Medication":
            for c in iter_codings(res.get("code")):
                add_coding(rt, "code", c)

        elif rt == "Immunization":
            for c in iter_codings(res.get("vaccineCode")):
                add_coding(rt, "vaccineCode", c)

coding_df = pd.DataFrame(coding_rows)

if not coding_df.empty:
    coding_system_summary = (
        coding_df.groupby(["resource_type","field","system"], dropna=False)
        .agg(coding_instances=("code","size"), unique_codes=("code","nunique"))
        .reset_index()
        .sort_values(["resource_type","coding_instances"], ascending=[True,False])
    )
else:
    coding_system_summary = pd.DataFrame(
        columns=["resource_type","field","system","coding_instances","unique_codes"]
    )

display(coding_system_summary)
coding_system_summary.to_csv(
    PUBLIC_OUT / "clean25_fhir_coding_system_summary.csv",
    index=False
)

coding_df.to_csv(
    PRIVATE_DIAG_DIR / "clean25_fhir_code_level_detail_PRIVATE.csv",
    index=False
)

,resource_type,field,system,coding_instances,unique_codes
0,Encounter,class,http://terminology.hl7.org/CodeSystem/v3-ActCode,1324,5
1,Encounter,type[0],http://snomed.info/sct,1324,35
2,Immunization,vaccineCode,http://hl7.org/fhir/sid/cvx,403,20
3,Medication,code,http://www.nlm.nih.gov/research/umls/rxnorm,1228,29
4,MedicationRequest,medicationCodeableConcept,http://www.nlm.nih.gov/research/umls/rxnorm,582,63


## 15. If Athena is loaded, resolve source concepts for unmapped drug/visit records

In [17]:
source_concept_resolution_rows = []

if ATHENA_VOCAB_DIR is not None and table_exists(conn, "concept"):
    for table in ["drug_exposure", "visit_occurrence"]:
        concept_col, source_value_col, source_concept_col = MAPPING_CONFIG[table]
        cols = columns_for(conn, table)
        if source_concept_col not in cols:
            continue

        detail = pd.read_sql_query(
            f'''SELECT
                    t."{source_concept_col}" AS source_concept_id,
                    COUNT(*) AS records,
                    c.concept_name,
                    c.domain_id,
                    c.vocabulary_id,
                    c.standard_concept,
                    c.invalid_reason
                FROM "{table}" t
                LEFT JOIN concept c
                  ON c.concept_id = t."{source_concept_col}"
                WHERE (t."{concept_col}" IS NULL OR t."{concept_col}" = 0)
                  AND t."{source_concept_col}" IS NOT NULL
                  AND t."{source_concept_col}" <> 0
                GROUP BY
                    t."{source_concept_col}",
                    c.concept_name, c.domain_id, c.vocabulary_id,
                    c.standard_concept, c.invalid_reason
                ORDER BY records DESC''',
            conn
        )
        detail["omop_table"] = table
        source_concept_resolution_rows.append(detail)

        detail.to_csv(
            PRIVATE_DIAG_DIR / f"{table}_unmapped_source_concept_resolution_PRIVATE.csv",
            index=False
        )

    if source_concept_resolution_rows:
        combined = pd.concat(source_concept_resolution_rows, ignore_index=True)
        public_resolution = (
            combined.groupby(
                ["omop_table","vocabulary_id","domain_id","standard_concept","invalid_reason"],
                dropna=False
            )
            .agg(
                distinct_source_concepts=("source_concept_id","nunique"),
                records=("records","sum")
            )
            .reset_index()
        )
        display(public_resolution)
        public_resolution.to_csv(
            PUBLIC_OUT / "clean25_unmapped_source_concept_aggregate.csv",
            index=False
        )
else:
    print("Athena was not loaded; source-concept vocabulary resolution is intentionally skipped.")

Athena was not loaded; source-concept vocabulary resolution is intentionally skipped.


# Part E — Practical performance and automated validation

## 16. Stage-level timing

In [18]:
timing_rows = [
    {"stage": "dependency_install", "seconds": INSTALL_SECONDS, "scope": "environment setup"},
    {"stage": "synthea_generation", "seconds": SYNTHEA_GENERATION_SECONDS, "scope": "25-patient synthetic data generation"},
    {"stage": "fhiry_normalization", "seconds": FHIRY_NORMALIZATION_SECONDS, "scope": "FHIR Bulk NDJSON -> pandas"},
]

if VOCAB_LOAD_SECONDS is not None:
    timing_rows.append({
        "stage": "athena_vocabulary_load",
        "seconds": VOCAB_LOAD_SECONDS,
        "scope": "Athena vocabulary initialization"
    })

timing_rows += [
    {"stage": "pyomop_transform", "seconds": PYOMOP_TRANSFORM_SECONDS, "scope": "normalized source -> fresh OMOP SQLite"},
    {"stage": "omop_load_total", "seconds": TOTAL_OMOP_LOAD_SECONDS, "scope": "vocabulary if present + transform"},
]

timing_df = pd.DataFrame(timing_rows)
timing_df["seconds"] = timing_df["seconds"].astype(float).round(4)
display(timing_df)
timing_df.to_csv(PUBLIC_OUT / "clean25_stage_timing.csv", index=False)

,stage,seconds,scope
0,dependency_install,4.8179,environment setup
1,synthea_generation,52.4128,25-patient synthetic data generation
2,fhiry_normalization,117.5805,FHIR Bulk NDJSON -> pandas
3,pyomop_transform,47.4731,normalized source -> fresh OMOP SQLite
4,omop_load_total,50.9708,vocabulary if present + transform


## 17. Automated reviewer-resolution checks

In [19]:
checks = []

def add_check(check_id, description, passed, observed, expected, note=""):
    checks.append({
        "check_id": check_id,
        "description": description,
        "passed": bool(passed),
        "observed": observed,
        "expected": expected,
        "note": note,
    })

add_check(
    "SRC-25",
    "Exactly 25 unique FHIR Patient resources",
    len(patient_unique) == 25 and len(patient_ids) == 25,
    f"{len(patient_ids)} resources / {len(patient_unique)} unique",
    "25 resources / 25 unique",
)

add_check(
    "DB-FRESH",
    "Target OMOP database was recreated from a deleted/non-existing file",
    DB_PATH.exists() and DB_PATH.stat().st_size > 0,
    f"created={DB_PATH.exists()}, size_mb={round(DB_PATH.stat().st_size/(1024**2),3)}",
    "new non-empty SQLite DB",
    "DB file was explicitly unlinked before pyOMOP initialization."
)

add_check(
    "PERSON-REC",
    "FHIR Patient count equals OMOP PERSON row count",
    person_count == len(patient_unique),
    person_count,
    len(patient_unique),
    "Core reviewer-resolution reconciliation check."
)

add_check(
    "VOCAB",
    "Athena vocabulary available for vocabulary-backed mapping reproduction",
    ATHENA_VOCAB_DIR is not None,
    str(ATHENA_VOCAB_DIR) if ATHENA_VOCAB_DIR else "not found",
    "Athena vocabulary directory",
    "Failure gates comparison to submitted vocabulary-backed mapping percentages."
)

for _, r in mapping_df.iterrows():
    add_check(
        f"MAP-{r['omop_table']}",
        f"Mapping completeness computed for {r['omop_table']}",
        pd.notna(r["mapped_percent"]),
        r["mapped_percent"],
        "numeric mapping percent",
        "Comparison to submitted percentages is valid only when Athena was loaded."
    )

checks_df = pd.DataFrame(checks)
display(checks_df)
checks_df.to_csv(
    PUBLIC_OUT / "clean25_automated_validation.csv",
    index=False
)

,check_id,description,passed,observed,expected,note
0,SRC-25,Exactly 25 unique FHIR Patient resources,True,25 resources / 25 unique,25 resources / 25 unique,
1,DB-FRESH,Target OMOP database was recreated from a dele...,True,"created=True, size_mb=6.039",new non-empty SQLite DB,DB file was explicitly unlinked before pyOMOP ...
2,PERSON-REC,FHIR Patient count equals OMOP PERSON row count,True,25,25,Core reviewer-resolution reconciliation check.
3,VOCAB,Athena vocabulary available for vocabulary-bac...,False,not found,Athena vocabulary directory,Failure gates comparison to submitted vocabula...
4,MAP-condition_occurrence,Mapping completeness computed for condition_oc...,True,0.0,numeric mapping percent,Comparison to submitted percentages is valid o...
5,MAP-drug_exposure,Mapping completeness computed for drug_exposure,True,0.0,numeric mapping percent,Comparison to submitted percentages is valid o...
6,MAP-measurement,Mapping completeness computed for measurement,True,0.0,numeric mapping percent,Comparison to submitted percentages is valid o...
7,MAP-observation,Mapping completeness computed for observation,True,0.0,numeric mapping percent,Comparison to submitted percentages is valid o...
8,MAP-visit_occurrence,Mapping completeness computed for visit_occurr...,True,0.0,numeric mapping percent,Comparison to submitted percentages is valid o...


# Part F — Reviewer-facing interpretation

## 18. Generate a conservative evidence summary

In [20]:
drug_matches = mapping_df[mapping_df["omop_table"] == "drug_exposure"]
visit_matches = mapping_df[mapping_df["omop_table"] == "visit_occurrence"]

drug_row = drug_matches.iloc[0] if not drug_matches.empty else None
visit_row = visit_matches.iloc[0] if not visit_matches.empty else None

summary = {
    "historical_25_to_27_cause": "not causally reconstructable from preserved original row-level assets",
    "clean_reproduction_fhir_patients": len(patient_unique),
    "clean_reproduction_omop_person_rows": person_count,
    "clean_reproduction_person_reconciled": person_count == len(patient_unique),
    "athena_loaded": ATHENA_VOCAB_DIR is not None,
    "drug_mapping_percent_clean_run": None if drug_row is None else float(drug_row["mapped_percent"]),
    "visit_occurrence_rows_clean_run": None if visit_row is None else int(visit_row["total_records"]),
    "visit_nonzero_concept_percent_clean_run": None if visit_row is None else float(visit_row["mapped_percent"]),
    "submitted_mapping_comparison_allowed": ATHENA_VOCAB_DIR is not None,
}

summary_df = pd.DataFrame([summary])
display(summary_df)
summary_df.to_csv(
    PUBLIC_OUT / "clean25_reviewer_resolution_summary.csv",
    index=False
)

if person_count == len(patient_unique):
    person_statement = (
        f"In a clean reviewer-resolution reproduction using a newly generated deterministic "
        f"{len(patient_unique)}-patient Synthea cohort and a freshly initialized SQLite OMOP database, "
        f"the workflow produced {person_count} PERSON rows, demonstrating exact source-to-target row-count "
        f"reconciliation in the current workflow. The original historical 25-to-27 discrepancy cannot be "
        f"assigned a specific cause because the original row-level source and target assets were not preserved."
    )
else:
    person_statement = (
        f"In the clean reviewer-resolution reproduction, {len(patient_unique)} FHIR Patients produced "
        f"{person_count} OMOP PERSON rows. Because this run preserves source and target assets, "
        f"the discrepancy is now directly diagnosable from this run."
    )

if visit_row is not None:
    visit_statement = (
        f"The clean run produced {int(visit_row['total_records']):,} visit_occurrence rows, of which "
        f"{float(visit_row['mapped_percent']):.2f}% had a non-zero visit_concept_id."
    )
else:
    visit_statement = "No visit_occurrence mapping summary was available."

if ATHENA_VOCAB_DIR is None:
    mapping_scope_statement = (
        "Athena vocabulary files were not available in this run, so vocabulary-backed mapping percentages "
        "from the submitted demonstration are not treated as reproduced or directly comparable."
    )
else:
    mapping_scope_statement = (
        "Athena vocabulary files were loaded in this run, so the clean-run mapping percentages are directly "
        "interpretable as vocabulary-backed mapping results under the recorded software/vocabulary environment."
    )

reviewer_text = (
    "PERSON reconciliation:\n"
    + person_statement
    + "\n\nVisit mapping:\n"
    + visit_statement
    + "\n\nMapping scope:\n"
    + mapping_scope_statement
)

print(reviewer_text)

(PUBLIC_OUT / "clean25_reviewer_ready_text.txt").write_text(
    reviewer_text,
    encoding="utf-8"
)

,historical_25_to_27_cause,clean_reproduction_fhir_patients,clean_reproduction_omop_person_rows,clean_reproduction_person_reconciled,athena_loaded,drug_mapping_percent_clean_run,visit_occurrence_rows_clean_run,visit_nonzero_concept_percent_clean_run,submitted_mapping_comparison_allowed
0,not causally reconstructable from preserved or...,25,25,True,False,0.0,1324,0.0,False


PERSON reconciliation:
In a clean reviewer-resolution reproduction using a newly generated deterministic 25-patient Synthea cohort and a freshly initialized SQLite OMOP database, the workflow produced 25 PERSON rows, demonstrating exact source-to-target row-count reconciliation in the current workflow. The original historical 25-to-27 discrepancy cannot be assigned a specific cause because the original row-level source and target assets were not preserved.

Visit mapping:
The clean run produced 1,324 visit_occurrence rows, of which 0.00% had a non-zero visit_concept_id.

Mapping scope:
Athena vocabulary files were not available in this run, so vocabulary-backed mapping percentages from the submitted demonstration are not treated as reproduced or directly comparable.


776

## 19. Final environment/reproducibility manifest

In [21]:
environment.update({
    "synthea_git_commit": synthea_sha,
    "synthea_seed": SYNTHEA_SEED,
    "synthea_reference_date": SYNTHEA_REFERENCE_DATE,
    "synthea_population": SYNTHEA_POPULATION,
    "synthea_overflow_population": False,
    "synthea_state": SYNTHEA_STATE,
    "fhir_bulk_dir_private": str(FHIR_BULK_DIR),
    "omop_db_private": str(DB_PATH),
    "athena_vocab_private": str(ATHENA_VOCAB_DIR) if ATHENA_VOCAB_DIR else None,
    "fhir_patient_count": len(patient_unique),
    "omop_person_count": person_count,
    "person_reconciliation_pass": person_count == len(patient_unique),
})

manifest_path = PUBLIC_OUT / "clean25_environment_manifest.json"
manifest_path.write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8"
)

print(json.dumps(environment, indent=2))

{
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "logical_cpu_count": 8,
  "physical_cpu_count": 4,
  "ram_gb": 50.99,
  "pyomop": "6.4.0",
  "fhiry": "5.2.2",
  "pandas": "2.2.2",
  "numpy": "2.0.2",
  "mapping_path": "/usr/local/lib/python3.12/dist-packages/pyomop/mapping.default.json",
  "mapping_sha256": "a35ea4afa6a516c68441d704af5ccc6ed25871b9b20ded8463f6f5b8eb820456",
  "dependency_install_seconds": 4.8179,
  "synthea_git_commit": "7e08387c68a7f0e21d13076609a159fd473fc902",
  "synthea_seed": 20260815,
  "synthea_reference_date": "20260815",
  "synthea_population": 25,
  "synthea_overflow_population": false,
  "synthea_state": "Massachusetts",
  "fhir_bulk_dir_private": "/content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/synthea_output/fhir",
  "omop_db_private": "/content/drive/MyDrive/fhir_omop_colab/reviewer_clean25/omop/reviewer_clean25.sqlite",
  "athena_vocab_private": null,
  "fhir_patient_coun

## 20. Close database and package GitHub-safe outputs

In [22]:
conn.close()

readme_text = (
    "# Clean 25-Patient Reviewer Resolution\n\n"
    "Generated by `OHDSI_Clean25_ReviewerResolution_COLAB.ipynb`.\n\n"
    "This directory contains aggregate reviewer-resolution evidence only.\n\n"
    "The run is a new deterministic clean reproduction and is **not** presented as the original "
    "historical 25-patient cohort.\n\n"
    "Raw FHIR NDJSON, the SQLite database, Athena vocabulary files, row-level identifiers, and "
    "code-level private diagnostics remain outside GitHub.\n"
)
(PUBLIC_OUT / "README.md").write_text(readme_text, encoding="utf-8")

archive = shutil.make_archive(
    "/content/OHDSI_clean25_reviewer_resolution_public_outputs",
    "zip",
    root_dir=str(PUBLIC_OUT)
)

print("Public output ZIP:", archive)
print("Public output directory:", PUBLIC_OUT)
print("Private run directory:", PRIVATE_ROOT)

print("\nGit status:")
subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"], check=False)

Public output ZIP: /content/OHDSI_clean25_reviewer_resolution_public_outputs.zip
Public output directory: /content/ohdsi-fhir-omop-showcase-demo/results/reviewer_clean25_resolution
Private run directory: /content/drive/MyDrive/fhir_omop_colab/reviewer_clean25

Git status:


CompletedProcess(args=['git', '-C', '/content/ohdsi-fhir-omop-showcase-demo', 'status', '--short'], returncode=0)